# 04 — Full Pipeline (Punctuation → Ordering → Grouping → Translation)

This notebook demonstrates the end-to-end pipeline implemented in the repository.

**Constraints:**
- No GPT/Gemini/Claude or other off-the-shelf generative chat LLMs.
- GPU memory must remain < 1.5 GB if GPU is used (current scripts run on CPU).
- Original data in `data/raw/samples/` is never modified; literal `NULL` markers are preserved.

**Note:** Punctuation and ordering quality must be reviewed manually. Execution success does not guarantee correctness.

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('CWD:', Path.cwd())

## 1. Pipeline stages and scripts

1. **Prepare input**: `python src/prepare_input.py`  
   `data/raw/samples/*.txt` → `data/model_input/*.txt` (preserves `NULL`)
2. **Punctuation restoration**: `python src/punctuation.py`  
   Uses `Horizon-Labs/punctuation-restoration-small` (quantized ONNX, CPU) → `data/punctuated/*.txt`
3. **Sentence ordering (candidate, all samples)**: `python src/order_sentences.py`  
   Embedding-based heuristic ordering → `data/ordered/<sample>.ordered.txt`, mapping, stats
4. **Grouping (5) + translation (Hindi→English)**: `python src/group_and_translate.py`  
   Groups every 5 consecutive ordered sentences and translates each group using `Helsinki-NLP/opus-mt-hi-en` (MarianMT seq2seq, CPU) → `data/final/*`

Run all steps at once: `python scripts/run_full_pipeline.py`

## 2. Inspect ordering outputs

In [ ]:
summary_path = Path('data/ordered/ordering_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    for f in summary['files']:
        print(Path(f['input']).name, '-> sentences:', f['sentences'], 'topics:', f['topics'])
else:
    print('No ordering_summary.json found. Run: python src/order_sentences.py')

sample = 'sample-1'
ordered = Path(f'data/ordered/{sample}.ordered.txt')
if ordered.exists():
    lines = [ln for ln in ordered.read_text(encoding='utf-8').splitlines() if ln.strip()]
    print(f'\n{sample} ordered sentence count:', len(lines))
    for i, ln in enumerate(lines[:5], 1):
        print(f'{i}. {ln[:140]}{"..." if len(ln)>140 else ""}')

## 3. Inspect grouping and translation outputs

For each sample, the pipeline writes:
- `data/final/<sample>.groups_hi.txt` — groups of 5 Hindi sentences (in candidate order)
- `data/final/<sample>.groups_en.txt` — translated English groups
- `data/final/<sample>.groups.tsv` — aligned Hindi/English groups
- `data/final/<sample>.pipeline.json` — counts and paths

In [ ]:
ts = Path('data/final/translation_summary.json')
if ts.exists():
    tsummary = json.loads(ts.read_text(encoding='utf-8'))
    print('Translation model:', tsummary.get('translation_model'))
    print('Group size:', tsummary.get('group_size'))
    for f in tsummary.get('files', []):
        name = Path(f['ordered_input']).name
        print(name, 'sentences=', f['sentences'], 'groups=', f['groups'])
else:
    print('No translation_summary.json found. Run: python src/group_and_translate.py')

In [ ]:
# Preview one sample's first Hindi and English groups
sample = 'sample-1'
hi = Path(f'data/final/{sample}.groups_hi.txt')
en = Path(f'data/final/{sample}.groups_en.txt')
if hi.exists() and en.exists():
    print('--- Hindi groups (preview) ---')
    print('\n'.join(hi.read_text(encoding='utf-8').splitlines()[:12]))
    print('\n--- English groups (preview) ---')
    print('\n'.join(en.read_text(encoding='utf-8').splitlines()[:12]))
else:
    print('Final group files not found for', sample)

## 4. Integrity check (NULL preservation, counts)

Run the verification script to confirm literal `NULL` counts are preserved across stages:

In [ ]:
# In a notebook you can run the script via subprocess to see the same report
import subprocess, sys
subprocess.run([sys.executable, 'scripts/verify_integrity.py'], check=False)

## 5. Important review notes

- **Punctuation quality:** review `data/punctuated/*.txt` manually (and vs `data/raw/samples/*` context). Check `NULL`-adjacent punctuation with `02_null_diagnostic.ipynb` / `scripts/null_diagnostic.py`.
- **Ordering:** candidate heuristic order; validate for logical/original sequence, not fluency.
- **Translation:** produced by `Helsinki-NLP/opus-mt-hi-en` (MarianMT, encoder-decoder MT model). Quality depends on punctuation/order and domain; review outputs in `data/final/*.groups_en.txt`.
- **Compliance:** no GPT/Gemini/Claude used. Keep total GPU memory < 1.5 GB if adapting to GPU; current scripts default to CPU.